# Regression with XGBoost 

### Importing the libraries

In [ ]:
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

from scipy.optimize import differential_evolution

### Import the dataset

In [ ]:
dataset = load_dataset("maharshipandya/spotify-tracks-dataset")
data = dataset["train"].to_pandas()

### Select features and target

In [ ]:
feature_columns = ["danceability",
    "energy",
    "instrumentalness",
    "liveness",
    "loudness",
    "speechiness",
    "tempo",
    "valence",
    "acousticness"
]
target_column = "popularity"

X = data[feature_columns].copy()
y = data[target_column].copy()

###  Training/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

### Training XGBoost on the Training set with K-Fold

In [ ]:
regressor = XGBRegressor()

regressor.fit(X_train, y_train)

### Predicting the Test set results

In [ ]:
y_pred = regressor.predict(X_test)

### Evaluate Model Performance

In [ ]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(
    mean_squared_error(y_test, y_pred)
)

r2 = r2_score(y_test, y_pred)

print("\nModel performance")
print("-------------------------")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

--------------
## Bonus Section - Optimal XGBoost Model

### Define K-Fold Cross-Validation

In [ ]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=0
)

### Define XGBoost model

In [ ]:
xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=0,
    n_jobs=-1
)

### Define parameters to optimise

In [ ]:
param_grid = {

    "n_estimators": [
        200,
        300,
        400,
        500,
        600
    ],

    "learning_rate": [
        0.01,
        0.03,
        0.05,
        0.1
    ], "max_depth": [
        3,
        4,
        5,
        6
    ],

    "min_child_weight": [
        1,
        3,
        5
    ],

    "subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],
 "colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "gamma": [
        0,
        0.1,
        0.3
    ]
}

### Randomized Search with K-Fold Cross-Validation

In [ ]:
search = RandomizedSearchCV(

    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=30,
    scoring="r2",
    cv=kf,
    random_state=0,
    n_jobs=-1,
    verbose=1
)

### Test the different XGBoost configurations on the Training set

In [ ]:
search.fit(
    X_train,
    y_train
)


### Display Best Parameters

In [ ]:
for parameter, value in search.best_params_.items():

    print(
        f"{parameter}: {value}"
    )


print("\nBest Cross-Validated R²:")
print(
    round(search.best_score_, 4)
)

### Select the best XGBoost model

In [ ]:
optimized_regressor = search.best_estimator_

### Predict the original Test set

In [ ]:
y_pred_optimized = optimized_regressor.predict(
    X_test
)

### Evaluate Optimised Model

In [ ]:
mae_optimized = mean_absolute_error(
    y_test,
    y_pred_optimized
)

rmse_optimized = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred_optimized
    )
)

r2_optimized = r2_score(
    y_test,
    y_pred_optimized
)

print(
    "MAE :",
    round(mae_optimized, 4)
)

print(
    "RMSE:",
    round(rmse_optimized, 4)
)

print(
    "R²  :",
    round(r2_optimized, 4)
)